
# Amazon ML Challenge 2026 - Business Entity Resolution (baseline)

**Goal:** for every test `S1` entity, predict the *set* of `S2`/`S3` records
that refer to the same real-world legal entity. Ranked at entity resolution
with **macro-F0.5** as primary metric.

**Approach** (runtime, CPU-only, ~16-32 GB RAM):

1. Multi-lingual normalization (Unicode preserved; auditable Devanagari->Latin
   fold used *only* to build blocking keys).
2. Blocking per country: TF-IDF (char-wb) top-k retrieval + exact `core_key`
   pass over S2/S3 to form candidate pairs (~25/S1 cap). **Never** the full
   S1 x (S2 U S3) Cartesian product.
3. Character/name/address/numeric features + LightGBM classifier.
4. Greedy prefix decision from calibrated probabilities + thresholds tuned on
   a validation fold, then a **global one-parent** wrap for S1 groups.
5. Outputs `matching_results.tsv` + `candidate_pairs.tsv`, checked with the
   official validator.

**Compliance guardrails:** no external datasets, no geocoding/registry, no
LLM embeddings; `country` is an open set (US / India / France processed
identically). The empty-non-singleton convention (A3) is **provisional
0.0** and marked as such - not claimed as the official organizer rule.

**Human-first:** every heavy stage is cached under `DATA_ROOT/cache/`; you can
re-open the notebook after a runtime disconnect and resume without
rebuilding. See `reports/notebook_execution_guide.md`.

> For a *quick* correctness pass set `SMOKE_MODE=True` below (uses a few
> thousand S1 rows) before the full run.



### Notebook index (21 sections)

| # | Section | # | Section |
|---|---|---|---|
| 1 | Environment & configuration | 11 | LightGBM training |
| 2 | Dataset paths | 12 | Threshold / set optimization |
| 3 | Dependencies | 13 | Global one-parent resolution |
| 4 | Dataset loading | 14 | Validation results |
| 5 | EDA / dataset summary | 15 | Train final model |
| 6 | Normalization | 16 | Test candidate generation |
| 7 | Validation split | 17 | Test inference |
| 8 | Candidate generation / blocking | 18 | Generate matching_results.tsv |
| 9 | Candidate recall evaluation | 19 | Generate candidate_pairs.tsv |
| 10 | Feature engineering | 20 | Validate submission |
| 11 | LightGBM training | 21 | Save / download submission files |



## 1. Environment & configuration

The pipeline auto-detects the runtime (Kaggle / Colab / local) and the data
folder. Override `DATA_ROOT` only if auto-detection picks the wrong place:

- **Kaggle:** after adding the dataset, Auto-Detect finds
  `/kaggle/input/<ds>/` for you. The `code/` folder must be uploaded to
  `/kaggle/working/`.
- **Colab:** upload `student_resource/` to Drive and set
  `DATA_ROOT = "/content/drive/MyDrive/<path>/student_resource"`, or use the
  upload cell in section 2. `code/` should live at `/content/code/`.
- **Local (dev only):** the code also runs on a laptop for smoke tests only -
  never run the heavy stages locally.

`SMOKE_MODE` (default **off**) bounds every country to a few thousand S1 rows
so the whole pipeline can be verified in ~5 minutes.


In [ ]:
# ---------- 1. Environment & configuration ------------------------------
import gc, glob, json, os, platform, shutil, subprocess, sys, time

CODE_ROOT_OVERRIDE = ""     # e.g. "/kaggle/working"  (defaults to os.getcwd())
DATA_ROOT_OVERRIDE = ""     # e.g. "/kaggle/input/mlchallenge2026-er"
SMOKE_MODE = False          # True -> quick per-country bound run for testing

CODE_ROOT = CODE_ROOT_OVERRIDE or os.getcwd()
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from code.business_entity_resolution.src import config as C
from code.business_entity_resolution.src import normalize as N
from code.business_entity_resolution.src import data as D
from code.business_entity_resolution.src import blocking as B
from code.business_entity_resolution.src import features as F
from code.business_entity_resolution.src import scoring as S
from code.business_entity_resolution.src import model as M
from code.business_entity_resolution.src import decision as DEC
from code.business_entity_resolution.src import pipeline as P

cfg = C.Config()
if SMOKE_MODE:                      # bound sizes for a fast end-to-end check
    cfg.smoke = True
    cfg.smoke_s1_limit = 4000
    cfg.smoke_pool_limit = 60000
    cfg.max_features = 100000
    cfg.validation_fraction = 0.05
    cfg.negative_sample_ratio = 1.0
# optional threshold grids (used by section 12 tuning)
cfg.threshold_grid_first = (0.90, 0.95, 0.98)
cfg.threshold_grid_add  = (0.95, 0.98, 1.00)
P.setup(cfg, DATA_ROOT_OVERRIDE or None)

print("\n== environment ==")
print("host           :", platform.node(), "|", platform.system(), platform.machine())
print("python         :", sys.version.split()[0])
print("DATA_ROOT      :", cfg.paths().data_root)
print("cache_dir      :", cfg.paths().cache_dir)
print("output_dir     :", cfg.paths().output_dir)
print("SMOKE_MODE     :", cfg.smoke)


In [ ]:
# ---------- 1b. Runtime resources (disk / RAM / GPU) ---------------------
def _disk(p):
    u = shutil.disk_usage(p)
    return f"{u.free / 2**30:.1f} GB free of {u.total / 2**30:.1f} GB"

print("disk /         :", _disk("/"))
try:
    import psutil
    m = psutil.virtual_memory()
    print(f"RAM            : {m.total / 2**30:.1f} GB total | {m.available / 2**30:.1f} GB available")
except Exception:
    print("RAM            : (psutil not installed - skip)")

try:
    out = subprocess.run(["nvidia-smi", "--query-gpu=name",
                          "--format=csv,noheader"], capture_output=True, text=True, timeout=30)
    gpu = out.stdout.strip() or "none detected"
except Exception:
    gpu = "none detected (CPU baseline)"
print("GPU            :", gpu, "(not required - CPU baseline)")
print("DATA_ROOT      :", cfg.paths().data_root)



## 2. Dataset paths

`DATA_ROOT` points to the folder that contains `dataset/` and `utils/`. The
notebook derives every path from it, so there are **no hard-coded absolute
paths** below. If the sample data lives elsewhere, edit `DATA_ROOT` in
section 1 and re-run this cell.

**Colab - upload your own data** (skip on Kaggle):

```python
from google.colab import files
import zipfile; z = zipfile.ZipFile(files.upload().popitem()[1]); z.extractall("/content/amazon_ml")
```


In [ ]:
# ---------- 2. Dataset paths -----------------------------------------------
pr = cfg.paths()
for split in ("train", "test"):
    folder = os.path.join(pr.data_root, "dataset", split)
    files_ = sorted(glob.glob(os.path.join(folder, "*.tsv")))
    print(split.upper(), "->", folder)
    for f_ in files_[:4]:
        n = os.path.basename(f_)
        sz = os.path.getsize(f_) / 2**20
        print(f"   {n:32s} {sz:8.1f} MB")
gt_file = os.path.join(pr.data_root, "dataset", "train_ground_truth.tsv")
print("GT  ->", gt_file, f"({os.path.getsize(gt_file)/2**20:.1f} MB)" if os.path.exists(gt_file) else "(missing)")
print("validator ->", os.path.join(pr.data_root, "utils", "validate_submission.py"))



## 3. Dependencies

Installed **inside the runtime only** (Kaggle / Colab). The laptop is the dev
environment and must not install these. Present on most Kaggle/Colab images,
so this is usually a no-op. `requirements.txt` in the `code/` folder records
the same set.

```text
pandas numpy scikit-learn scipy lightgbm rapidfuzz pyarrow tqdm
```


In [ ]:
%%capture --no-display
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "pandas", "numpy", "scikit-learn", "scipy",
                       "lightgbm", "rapidfuzz", "pyarrow", "tqdm"])
print("dependencies installed")

In [ ]:
# ---------- 3b. Version check ---------------------------------------------
import pandas as pd, numpy as np, sklearn, scipy, lightgbm, rapidfuzz, pyarrow
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("scipy       ", scipy.__version__)
print("lightgbm    ", lightgbm.__version__)
print("rapidfuzz   ", rapidfuzz.__version__)
print("pyarrow     ", pyarrow.__version__)
# optional smoke import of the modules that (lazily) depend on rapidfuzz
import importlib
importlib.import_module("code.business_entity_resolution.src.features")



## 4. Dataset loading

Reads the 7 entity TSVs (S1/S2/S3 of train and test). Every entity gets:

- **raw** columns preserved exactly (no destructive normalization),
- `norm_*` Unicode NFKC-normalized views,
- `latn_*` Latin folds used only as **search keys** in blocking,
- `core_key`, numeric/postal token strings.

The result is cached as parquet under `cache/`, so the *second* time you run
the notebook (e.g. after a disconnect) this is instant. Expect ~8-14 GB
transient RAM on the full dataset.


In [ ]:
# ---------- 4. Dataset loading --------------------------------------------
t0 = time.time()
train_tabs = P.build_entities(cfg, ("train",))
test_tabs  = P.build_entities(cfg, ("test",))
print("entities cached under:", cfg.paths().cache_dir)
for split, tabs in (("train", train_tabs), ("test", test_tabs)):
    for kind, tab in tabs.items():
        print(f"{split}/{kind}: {len(tab):,} rows -> "
              f"cache/sample_{split}_s{kind}.parquet")
print(f"loaded in {time.time()-t0:.1f}s")
# free the raw (un-normalized) copies to keep RAM low later
for tabs in (train_tabs, test_tabs):
    for kind, tab in tabs.items():
        tab.drop(columns=[c for c in ("raw_name", "raw_addr") if c in tab.columns],
                 errors="ignore", inplace=True)
gc.collect()



## 5. EDA / dataset summary

Reuses findings from `reports/eda.md` (Phase 0) and prints a live summary
from the cached entities: record counts, null rates, S1 group sizes, country
breakdown, and the S1-level match distribution (singleton fraction).


In [ ]:
# ---------- 5. EDA / dataset summary ---------------------------------------
print(P.entity_summary(cfg).to_string(index=False))



## 6. Normalization

The pipeline **preserves Unicode** (raw + normalized views). Devanagari is
folded to Latin with a *manual, auditable character table* (see
`normalize.py:_DEVA_TO_LATIN`) because it is only used for blocking keys -
never stored as the final "identity" of a record. Hindi orthography drops the
inherent vowel, so folded search keys are approximate (`globl` vs `global`);
TF-IDF n-grams + the exact core-key pass mitigate this.


In [ ]:
# ---------- 6. Normalization examples --------------------------------------
print("raw       | norm      | latn (fold) | core_key")
for raw in ["McDonald\u2019s Corp",
            "\u0917\u094d\u0932\u094b\u092c\u0932 \u0907\u0928\u094d\u0935\u0947\u0938\u094d\u091f\u092e\u0947\u0902\u091f",
            "ACME & SUM (L.L.C.)",
            "Rue des \u00c9glises 5"]:
    print(f"{raw:48s} | {N.normalize_text(raw):12s} | {N.latin_fold(raw):28s} | {N.core_key(raw)}")
assert "\u0907\u0928\u094d" not in N.normalize_text("\u0917\u0932\u094b\u092c\u0932"), "Devanagari should be preserved in unicode view"
assert "and" in N.latin_fold("ACME & Sum"), "ampersand fold failed"
print("normalization sanity: OK  (devanagari preserved in unicode view)")



## 7. Validation split

Ground-truth is loaded and the **train** S1 set split into train/validation
folds **grouped by S1** (no S1 leaks between folds). S2/S3 are not split -
blocking happens inside the train pool and the fold only restricts which S1
groups are scored.


In [ ]:
# ---------- 7. Validation split ---------------------------------------------
gt_map = P.build_gt(cfg)                       # S1 -> set(children)
train_s1_ids = train_tabs["s1"]["entity_id"].astype(str).tolist()
train_ids, valid_ids = P.group_split(cfg, train_s1_ids)
print(f"train S1: {len(train_ids):,}  |  validation S1: {len(valid_ids):,} "
      f"({len(valid_ids)/len(train_ids)*100:.2f}%)")
import collections
cnt = collections.Counter(train_tabs["s1"].set_index("entity_id").loc[train_ids, "country"])
print("train folds by country:", dict(cnt))
gt_sizes = [len(v) for v in gt_map.values()]
print(f"ground truth: {len(gt_map):,} S1 groups | "
      f"mean {np.mean(gt_sizes):.2f} matches | singletons {100*np.mean(np.array(gt_sizes)==0):.2f}%")



## 8. Candidate generation / blocking

Per country (open set - France handled exactly like US/India), build a
TF-IDF (char_wb 3..5-gram) index over S2 U S3 and retrieve the top-k nearest
neighbors for every S1, then add an **exact core-key** pass. Results are
ranked/deduped and capped at `max_candidates_per_s1` (default 25). Suitable
for the ~26M-record dataset:

- TF-IDF matrix float32, built **per country** (bounded in RAM),
- chunked dense scoring (`query_chunk` rows at a time),
- top-k selection via `np.argpartition` (no full pairwise matrix).

Candidates are cached to parquet for resumability.


In [ ]:
# ---------- 8. Candidate generation / blocking (train) --------------------
def _build_train_candidates():
    cands_train, cands_valid, _bst = P.candidates_train(
        cfg, train_tabs, gt_map, train_ids, valid_ids)
    return cands_train, cands_valid, _bst

tr_p = os.path.join(cfg.paths().cache_dir, "cands_train.parquet")
vl_p = os.path.join(cfg.paths().cache_dir, "cands_valid.parquet")
if cfg.smoke or not (os.path.isfile(tr_p) and os.path.isfile(vl_p)):
    t0 = time.time()
    cands_train, cands_valid, block_stats = _build_train_candidates()
    cands_train.to_parquet(tr_p, index=False)
    cands_valid.to_parquet(vl_p, index=False)
    print(f"blocking built in {time.time()-t0:.1f}s -> cached")
else:
    cands_train = pd.read_parquet(tr_p)
    cands_valid = pd.read_parquet(vl_p)
    block_stats = {}   # recomputed by scoring in section 9
    print("blocking loaded from cache")
print(f"train candidates: {len(cands_train):,} rows | validation candidates: {len(cands_valid):,}")
print("avg candidates/S1:", np.round(cands_train.groupby('s1', observed=True).size().mean(), 2))



## 9. Candidate recall evaluation

Reports blocking recall (proportion of GT pairs present among the candidates)
on the validation groups - overall and per country. The train-pool Golden
Target matches reach ~ (design target 0.98+); a low value would push you to
raise `block_top_k` / `max_features`.


In [ ]:
# ---------- 9. Candidate recall --------------------------------------------
s1c = P._country_map(cfg)
for name, cand, idset in (("validation (train pool)", cands_valid, valid_ids),):
    r, found, total, n_sing = B.blocking_recall(cand, gt_map, s1_ids=idset)
    print(f"[{name}] blocking recall = {r:.4f}  ({found}/{total} GT pairs)  "
          f"non-singleton S1 = {n_sing}")
    for ctry in sorted({s1c.get(i, '?') for i in idset}):
        ids_c = [i for i in idset if s1c.get(i) == ctry]
        rc, _, tc, _ = B.blocking_recall(cand, gt_map, s1_ids=ids_c)
        print(f"    {ctry:8s}: recall {rc:.4f} ({tc} GT pairs, {len(ids_c):,} S1)")



## 10. Feature engineering

Two passes over the candidate pair universe:

- **blocking features** - score/rank/best-gap/count (from retrieval),
- **similarity features** - normalized name/address ratios, token & char-based
  overlaps (RapidFuzz), numeric/postal consistency, country-equal.

The validation fold gets no labels. Training rows are down-sampled to a
balanced writer/S2-S3 negative pool *per S1* (see `negative_sample_ratio`).
The full-labeled frame is used only for the final-model retrain.


In [ ]:
# ---------- 10. Feature engineering ----------------------------------------
t0 = time.time()
df_train, X_train, feature_names = P.build_features(cfg, cands_train, train_tabs)
df_train = P.add_train_labels(cfg, df_train, gt_map)
df_valid, X_valid, _names2 = P.build_features(cfg, cands_valid, train_tabs)
assert _names2 == feature_names, "feature columns diverged between folds"
print(f"features built in {time.time()-t0:.1f}s")
print(f"{len(df_train):,} train rows x {len(feature_names)} features | "
      f"{len(df_valid):,} validation rows")
print(df_train[feature_names + ["y"]].head(3).to_string())
""



## 11. LightGBM training

A single LGBMClassifier over the feature vectors (binomial, row-wise
parallel). No GPU needed. The pairs are the samples - not the S1 groups - so
learning is per-candidate; the S1-group structure is imposed later by the
decision + one-parent layer. Down-sampling inside `train_model` keeps the
training frame balanced and small.


In [ ]:
# ---------- 11. LightGBM training ------------------------------------------
t0 = time.time()
model = P.train_model(cfg, df_train, feature_names)
print(f"trained in {time.time()-t0:.1f}s")
print("model type:", type(model).__module__ + "." + type(model).__name__)
""



## 12. Threshold / set optimization

On the validation fold we produce the greedy-prefix prediction under the
`cfg.threshold_grid_first` x `cfg.threshold_grid_add` combinations and pick
the macro-F0.5 best (the grid comes from the config set in section 1) -
reported in the table below. These tuned thresholds are then used for the
test decision.


In [ ]:
# ---------- 12. Threshold tuning + validation report ----------------------
result = P.validate_fold(cfg, model, df_valid, X_valid, gt_map,
                          valid_ids, feature_names)
best_first, best_add = result["best_first"], result["best_add"]
print(f"\nbest thresholds  first={best_first:.2f}  add={best_add:.2f} "
      f"=> macro-F0.5 = {result['report']['macro_f05']:.4f}")
""



## 13. Global one-parent resolution

Rule A1 (one parent per S2/S3 record) is **confirmed** by EDA. The decision
layer therefore keeps the globally most-confident parent per S2/S3. This cell
quantifies the gain over a raw per-S1 decision (no global tie-break).


In [ ]:
# ---------- 13. One-parent effect -------------------------------------------
dfv = P.predict_valid(cfg, model, df_valid, X_valid)
accepted_raw = DEC.decision_ids_from_frame(dfv, best_first, best_add,
                                           max_pred=cfg.max_pred_per_s1)
single = DEC.sets_from_frame(accepted_raw)
accepted_op = DEC.apply_one_parent(accepted_raw) if cfg.use_one_parent else accepted_raw
with_parent = DEC.sets_from_frame(accepted_op)
base_gt = {s: gt_map.get(s, set()) for s in dfv["s1"] if s in gt_map}
rep_a = S.metrics_report(base_gt, single, empty_non_singleton_score=cfg.empty_non_singleton_score)
rep_b = S.metrics_report(base_gt, with_parent, empty_non_singleton_score=cfg.empty_non_singleton_score)
print(f"without one-parent: macro-F0.5 {rep_a['macro_f05']:.4f}  (P {rep_a['precision']:.4f} R {rep_a['recall']:.4f})")
print(f"with one-parent   : macro-F0.5 {rep_b['macro_f05']:.4f}  (P {rep_b['precision']:.4f} R {rep_b['recall']:.4f})")
print("one-parent keeps the highest-probability parent per S2/S3 (A1 confirmed).")
""



## 14. Validation results

Full reported metrics: **macro-F0.5** (primary), precision, recall, singleton
accuracy, non-singleton recall, blocking recall, avg candidates/S1, avg
predictions/S1 and reduction ratio - overall and per country. The A3
convention (empty prediction on a non-singleton counts as F0.5=0) is
**provisional**; see `reports/eda.md`.


In [ ]:
# ---------- 14. Validation results (final table) ---------------------------
rep = result["report"]
print(S.format_report(rep))
print("\n-- per country --")
for ctry in sorted(rep["per_country"].keys()):
    c = rep["per_country"][ctry]
    print(f"{ctry:10s} macro-F0.5 {c['macro_f05']:.4f} | P {c['precision']:.4f} "
          f"| R {c['recall']:.4f} | singleton_acc {c['singleton_acc']:.3f} "
          f"| non-singleton_recall {c['non_singleton_recall']:.3f} "
          f"| {c['n_s1']:,} S1")
""



## 15. Train final model

A final LightGBM is fit on **all** train S1 groups (train fold + validation
fold) to use every label, then used for test inference. Thresholds are kept
from section 12. Heavy tables are dropped after this step to free RAM for the
test blocking stage.


In [ ]:
# ---------- 15. Train final model ------------------------------------------
RUN_FINAL_MODEL = True
if RUN_FINAL_MODEL:
    df_valid_with = P.add_train_labels(cfg, df_valid.copy(), gt_map)
    df_all = pd.concat([df_train, df_valid_with], ignore_index=True)
    t0 = time.time()
    final_model = P.train_model(cfg, df_all, feature_names)
    print(f"final model trained in {time.time()-t0:.1f}s on {len(df_all):,} rows")
else:
    final_model = model
# free memory for the test-pass
for _df in (df_train, X_train, df_valid, X_valid, df_all):
    try: del _df
    except Exception: pass
del train_tabs; gc.collect()
print("memory freed before test blocking")
""



## 16. Test candidate generation

Same blocking as section 8 but against the **test** S1 against the test S2/S3
pool. Cached for resumability. France S1 are included as normal country keys -
zero-shot by design (model was trained on US/India imagery only).


In [ ]:
# ---------- 16. Test candidate generation ----------------------------------
t0 = time.time()
tp_p = os.path.join(cfg.paths().cache_dir, "cands_test.parquet")
def _build_test_candidates():
    cands_test, _pool = P.candidates_test(cfg, test_tabs)
    return cands_test
if cfg.smoke or not os.path.isfile(tp_p):
    cands_test = _build_test_candidates()
    cands_test.to_parquet(tp_p, index=False)
    print(f"test blocking in {time.time()-t0:.1f}s -> cached")
else:
    cands_test = pd.read_parquet(tp_p)
    print("test candidates loaded from cache")
print(f"test candidates: {len(cands_test):,} rows | "
      f"avg/S1 {cands_test.groupby('s1', observed=True).size().mean():.2f}")
""



## 17. Test inference

Features are rebuilt for the test candidate pairs and the final model
predicts match probabilities. This is the last model-dependent step; the
output frame has one row per (S1, S2/S3) candidate.


In [ ]:
# ---------- 17. Test inference ---------------------------------------------
t0 = time.time()
df_test, _names_t = P.infer_test(cfg, final_model, cands_test, test_tabs)
assert _names_t == feature_names, "feature mismatch between train/test"
print(f"inferred in {time.time()-t0:.1f}s | {len(df_test):,} rows")
print(df_test.head(3).to_string())
""



## 18. Generate `matching_results.tsv`

Decisions: greedy-prefix over ranked candidates, `first>=best_first`,
then `add>=best_add`, capped at `max_pred_per_s1`, plus the global
one-parent wrap. Written to `<DATA_ROOT>/output/matching_results.tsv`.


In [ ]:
# ---------- 18. matching_results.tsv ---------------------------------------
test_s1_ids = test_tabs["s1"]["entity_id"].astype(str).tolist()
matches = P.decide_test(cfg, df_test, test_s1_ids,
                        first_threshold=best_first, add_threshold=best_add)
print(f"{len(matches):,} predicted S2/S3 across {len(test_s1_ids):,} test S1")
print("singleton prediction rate: "
      f"{100*np.mean([len(v)==0 for v in matches.values()]):.2f}%")
matching_path, _cand_path = P.write_outputs(cfg, matches, cands_test, test_tabs, write_candidates=False)
""



## 19. Generate `candidate_pairs.tsv`

The **full blocking candidate set** (matches results must be a strict subset).
Written to `<DATA_ROOT>/output/candidate_pairs.tsv` - used by the validator.


In [ ]:
# ---------- 19. candidate_pairs.tsv ----------------------------------------
_, candidate_path = P.write_outputs(cfg, matches, cands_test, test_tabs,
                                    write_matches=False)
print("candidate_pairs.tsv:", candidate_path)
""



## 20. Validate submission

Runs the official `utils/validate_submission.py` against both files.
- `check_ids=False` (default): moderate run.
- `check_ids=True`: verifies every S2/S3 id exists - loads the full test
  tables again (a few GB); run it once before submitting.
Prints `VALIDATOR: PASS/FAIL`.


In [ ]:
# ---------- 20. Validate submission ---------------------------------------
RUN_ID_CHECK = False
ok, out = P.run_validator(cfg, matching_path, candidate_path,
                          check_ids=RUN_ID_CHECK, verbose=True)
print("VALIDATOR:", "PASS" if ok else "FAIL")
assert ok, "validator failed - inspect the output above before continuing"
""



## 21. Save / download submission files

Zips `output/`, the pipeline `code/` and the documentation template into
`submission/preview_package.zip` (excludes the dataset/cache). On Colab it
also triggers the browser download. Final packaging per `PS.md` happens after
this notebook + `Documentation_template.md` are final.


In [ ]:
# ---------- 21. Save / download ---------------------------------------------
import zipfile
pkg = os.path.join(cfg.paths().data_root, "submission", "preview_package.zip")
os.makedirs(os.path.dirname(pkg), exist_ok=True)
with zipfile.ZipFile(pkg, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(cfg.paths().output_dir, "*")):
        z.write(f, os.path.relpath(f, cfg.paths().data_root))
    base = os.path.join(pr.data_root, "Documentation_template.md")
    if os.path.exists(base):
        z.write(base, os.path.relpath(base, pr.data_root))
    croot = os.path.join(pr.data_root, "code", "business_entity_resolution")
    if os.path.isdir(croot):
        for root, _, fs in os.walk(croot):
            if "__pycache__" in root: continue
            for fn in fs:
                pth = os.path.join(root, fn)
                z.write(pth, os.path.relpath(pth, pr.data_root))
print("package:", pkg, f"({os.path.getsize(pkg)/2**30:.2f} GiB)")
try:
    from google.colab import files
    files.download(pkg); print("download triggered (Colab)")
except Exception:
    print("not on Colab - download via file browser")
""



## Experiment summary & reproducibility

Filled by the code cell below (config dump + versions + key metrics) so the
experiment log in `Documentation_template.md` has exact numbers.


In [ ]:
# ---------- Experiment summary ---------------------------------------------
metrics = {
    "blocking_recall": block_stats.get("recall", float("nan")),
    "macro_f05_val": rep.get("macro_f05", float("nan")),
    "precision_val": rep.get("precision", float("nan")),
    "recall_val": rep.get("recall", float("nan")),
    "singleton_acc_val": rep.get("singleton_acc", float("nan")),
    "non_singleton_recall_val": rep.get("non_singleton_recall", float("nan")),
    "best_first": best_first,
    "best_add": best_add,
    "n_test_S1": len(test_s1_ids),
    "n_predicted_pairs": len(matches),
    "validation_fraction": cfg.validation_fraction,
}
print(json.dumps(metrics, indent=2))
print("\npipeline versions:", json.dumps(
    {"pandas": pd.__version__, "numpy": np.__version__,
     "scikit-learn": sklearn.__version__, "scipy": scipy.__version__,
     "lightgbm": lightgbm.__version__, "rapidfuzz": rapidfuzz.__version__}, indent=2))
""



## Future improvements (marked, not implemented in this baseline)

- FastText/BERT *offline* name embeddings (blocking reuse + feature) - would
  need extra storage; skipped to keep the pipeline dependency-light.
- Reranker (PWL/GBM pedestrian) on the top-k shortlist.
- Country-specific precision/recall weighting if the official F0.5 variant
  differs (A3 convention still provisional).
- Post-hoc one-parent refinement that also removes lower-priority S2 when an
  exact duplicate S3 exists.

Run order + failure recovery: see `reports/notebook_execution_guide.md`.
Notes on the design: see `DATASET_ANALYSIS.md` and `reports/eda.md`.
